# 05. Hybrid Recommendation Engine
Notebook ini membangun prototipe *Decision Support System* sejati untuk HRSS. Sistem menggabungkan:
1. **Machine Learning Model:** Mendeteksi keseluruhan profil operasional (*Standard* vs *Optimized*).
2. **Domain Knowledge Rule-Engine:** Menganalisis parameter mekanis & kelistrikan spesifik (gesekan, tegangan, aktivitas rel) untuk memberikan wawasan yang bisa langsung ditindaklanjuti.

In [1]:
import pandas as pd
import numpy as np
import pickle
import json

### 1. Desain Arsitektur Hybrid `HRSSHybridRecommender`
Kelas ini memiliki *Decision Policy Layer* yang memadukan keluaran probabilitas ML dengan status *Current Mode* dari mesin, serta mengeksekusi *Rule Engine*.

In [2]:
class HRSSHybridRecommender:
    def __init__(self, model_path, ml_threshold=0.5):
        # 1. Load ML Model
        with open(model_path, 'rb') as f:
            self.model = pickle.load(f)
        self.ml_threshold = ml_threshold
        
        # 2. Threshold untuk Domain Knowledge Rules
        # Nilai ini bisa di-tuning lebih lanjut berdasarkan distribusi persentil data pabrik
        self.rules_config = {
            'rail_activity_high': 0.05,        
            'power_efficiency_low': 0.01,      
            'total_power_high': 15.0,          
            'total_movement_low': 0.02,        
            'avg_voltage_drop': 23.5           # Standar drop tegangan pada bus DC 24V
        }
        
        self.expected_features = [
            'Timestamp', 'I_w_BLO_Weg', 'O_w_BLO_power', 'O_w_BLO_voltage', 'I_w_BHL_Weg', 'O_w_BHL_power', 'O_w_BHL_voltage', 'I_w_BHR_Weg', 'O_w_BHR_power', 'O_w_BHR_voltage', 'I_w_BRU_Weg', 'O_w_BRU_power', 'O_w_BRU_voltage', 'I_w_HR_Weg', 'O_w_HR_power', 'O_w_HR_voltage', 'I_w_HL_Weg', 'O_w_HL_power', 'O_w_HL_voltage', 'total_power', 'avg_voltage', 'total_movement', 'power_efficiency_ratio', 'rail_activity', 'conveyor_activity'
        ]

    def _preprocess(self, raw_data_dict):
        df = pd.DataFrame([raw_data_dict])
        for col in self.expected_features:
            if col not in df.columns:
                df[col] = 0.0
        return df[self.expected_features]
        
    def _evaluate_rules(self, df):
        '''Domain Knowledge Engine: Evaluasi anomali mekanis dan kelistrikan spesifik HRSS.'''
        alerts = []
        
        # Rule 1: Rail Inefficiency (Banyak gerak, efisiensi minim)
        if df['rail_activity'].values[0] > self.rules_config['rail_activity_high'] and df['power_efficiency_ratio'].values[0] < self.rules_config['power_efficiency_low']:
            alerts.append("Rail Inefficiency: Aktivitas sumbu horizontal tinggi namun rasio efisiensi daya sangat rendah. Pertimbangkan optimasi algoritma routing (shortest path) pada WMS.")
            
        # Rule 2: Mechanical Friction / Overload
        if df['total_power'].values[0] > self.rules_config['total_power_high'] and df['total_movement'].values[0] < self.rules_config['total_movement_low']:
            alerts.append("Overload/Friction Warning: Tarikan total arus/daya tinggi tanpa pergerakan mekanis yang proporsional. Lakukan inspeksi keausan guide rail atau cek batas maksimum muatan hoist.")
            
        # Rule 3: Electrical Voltage Drop (DC Bus)
        if df['avg_voltage'].values[0] < self.rules_config['avg_voltage_drop']:
            alerts.append("Voltage Sag Anomaly: Tegangan rata-rata turun di bawah batas aman. Hindari memberikan command akselerasi serentak pada multi-sumbu (X dan Y axis) untuk mencegah trip/reset sistem.")
            
        return alerts

    def _decision_policy_layer(self, current_mode, ml_probability, rule_alerts):
        '''Menggabungkan probabilitas ML, status mesin, dan Rules menjadi Rekomendasi Preskriptif.'''
        is_optimized_pred = ml_probability >= self.ml_threshold
        predicted_mode = "Optimized" if is_optimized_pred else "Standard"
        
        # Identifikasi selisih kondisi (Switch Logic)
        switch_recommended = (current_mode != predicted_mode)
        
        # Pemetaan Tingkat Risiko (Risk Level Mapping)
        if not switch_recommended and len(rule_alerts) == 0:
            risk_level = "Low Risk (Normal Operation)"
            primary_action = f"Kondisi beban ideal. Pertahankan mesin pada mode {current_mode}."
            
        elif switch_recommended and len(rule_alerts) == 0:
            risk_level = "Medium Inefficiency"
            primary_action = f"Pola kelistrikan cocok untuk mode {predicted_mode}. Disarankan melakukan transisi mode untuk meningkatkan efisiensi operasional."
            
        else:
            # Jika ada peringatan mekanis dari Rule Engine
            risk_level = "High Inefficiency / Mechanical Anomaly"
            primary_action = "Terdeteksi anomali beban fisik. Sistem menyarankan beralih ke mode Standard (jika saat ini berada di Optimized) untuk melindungi motor servo dari panas/aus."
            
        return risk_level, predicted_mode, primary_action

    def recommend(self, raw_data_dict, current_mode):
        '''Fungsi utama (Endpoint tiruan) yang memproses data dan mengeluarkan output JSON.'''
        # 1. Pipeline ML
        processed_df = self._preprocess(raw_data_dict)
        probability = self.model.predict_proba(processed_df)[0][1]
        
        # 2. Pipeline Domain Rules
        rule_alerts = self._evaluate_rules(processed_df)
        
        # 3. Penggabungan (Decision Policy)
        risk_level, predicted_mode, primary_action = self._decision_policy_layer(current_mode, probability, rule_alerts)
        
        return {
            "current_machine_mode": current_mode,
            "ml_predicted_profile": predicted_mode,
            "probability_optimized": f"{probability:.2%}",
            "operational_risk_level": risk_level,
            "primary_recommendation": primary_action,
            "technical_alerts": rule_alerts if rule_alerts else ["Semua metrik mekanis dan kelistrikan HRSS berada dalam batas normal."]
        }

### 2. Pengujian (End-to-End Simulation)
Kita akan mensimulasikan tiga skenario input berbeda dari sensor untuk menguji respon *Decision Policy Layer*.

In [3]:
# Inisialisasi engine hybrid
engine = HRSSHybridRecommender(model_path="../../models/final_hrss_rf_model.pkl", ml_threshold=0.5)

# Load data sampel
test_data = pd.read_csv("../data/splits/X_test_eng.csv")

# Skenario 1: Data Normal
sample_normal = test_data.iloc[42].to_dict()

# Skenario 2: Anomali Beban Berat (Modifikasi agar memicu Rule 2)
sample_anomaly = test_data.iloc[50].copy()
sample_anomaly['total_power'] = 20.5  # Daya sangat tinggi
sample_anomaly['total_movement'] = 0.01  # Pergerakan statis/sangat lambat
sample_anomaly = sample_anomaly.to_dict()

print("==================================================")
print("🔍 SIMULASI SKENARIO 1: Mesin Standard, Data Normal")
print("==================================================")
res1 = engine.recommend(sample_normal, current_mode="Standard")
print(json.dumps(res1, indent=4))

print("\n==================================================")
print("🚨 SIMULASI SKENARIO 2: Mesin Optimized, Terdeteksi Gesekan Tinggi")
print("==================================================")
res2 = engine.recommend(sample_anomaly, current_mode="Optimized")
print(json.dumps(res2, indent=4))

🔍 SIMULASI SKENARIO 1: Mesin Standard, Data Normal
{
    "current_machine_mode": "Standard",
    "ml_predicted_profile": "Optimized",
    "probability_optimized": "100.00%",
    "operational_risk_level": "Medium Inefficiency",
    "primary_recommendation": "Pola kelistrikan cocok untuk mode Optimized. Disarankan melakukan transisi mode untuk meningkatkan efisiensi operasional.",
    "technical_alerts": [
        "Semua metrik mekanis dan kelistrikan HRSS berada dalam batas normal."
    ]
}

🚨 SIMULASI SKENARIO 2: Mesin Optimized, Terdeteksi Gesekan Tinggi
{
    "current_machine_mode": "Optimized",
    "ml_predicted_profile": "Standard",
    "probability_optimized": "0.00%",
    "operational_risk_level": "High Inefficiency / Mechanical Anomaly",
    "primary_recommendation": "Terdeteksi anomali beban fisik. Sistem menyarankan beralih ke mode Standard (jika saat ini berada di Optimized) untuk melindungi motor servo dari panas/aus.",
    "technical_alerts": [
        "Overload/Friction Wa

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
